In [2]:
from ultralytics import YOLO

DATA_YAML = r"C:\Users\EPSRG\Documents\Iftee\hostpot_detection\hotspot_dataset\data.yaml"

def main():
    # Start from COCO-pretrained YOLO26-small. "s" gives more capacity than
    # "n" to pick up subtle low-contrast thermal signatures, while still
    # training fast on a 4070 at 512px.
    model = YOLO("yolo26s.pt")

    model.train(
        data=DATA_YAML,
        epochs=150,          # upper bound; patience will stop it earlier
        patience=30,         # stop if val metric hasn't improved in 30 epochs
        imgsz=512,            # matches your native thermal resolution
        batch=16,             # plenty of headroom on 12GB at 512px; raise to
        
        device=0,
        optimizer="MuSGD",    # YOLO26's native optimizer
        lr0=0.001,            # lower LR — appropriate for a ~574-image dataset
        cos_lr=True,

        # Small dataset (574 imgs): tone down the heaviest augmentations
        # that were designed for COCO-scale data.
        mosaic=0.5,
        mixup=0.0,
        copy_paste=0.0,
        close_mosaic=10,      # disable mosaic for the last 10 epochs

        name="yolo26s_run1",
        plots=True,
        val=True,
        seed=0,
    )

    # Quick look at test-set performance right after training (default conf=0.25)
    model.val(data=DATA_YAML, split="test")


if __name__ == "__main__":
    main()

Ultralytics 8.4.160  Python-3.14.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\EPSRG\Documents\Iftee\hostpot_detection\hotspot_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=0.5, multi_sca

In [3]:
"""
Sweep confidence thresholds on the test set and report precision/recall
at each, so you can pick an operating point deliberately rather than
using the default conf=0.25.

Since this project prioritizes recall (catch every hotspot, some false
alarms acceptable), look for the highest threshold that still keeps
recall above your target (e.g. >= 0.90), and use that conf value in
infer_hotspot.py.

Run:
    python evaluate_thresholds.py
"""

from ultralytics import YOLO

DATA_YAML = r"C:\Users\EPSRG\Documents\Iftee\hostpot_detection\hotspot_dataset\data.yaml"
WEIGHTS = r"C:\Users\EPSRG\Documents\Iftee\hostpot_detection\runs\detect\yolo26s_run1\weights\best.pt"

THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]


def main():
    model = YOLO(WEIGHTS)

    print(f"{'conf':>6} | {'precision':>9} | {'recall':>7} | {'mAP50':>6} | {'mAP50-95':>8}")
    print("-" * 50)

    for conf in THRESHOLDS:
        metrics = model.val(data=DATA_YAML, split="test", conf=conf, plots=False, verbose=False)
        p = metrics.box.mp      # mean precision across classes
        r = metrics.box.mr      # mean recall across classes
        map50 = metrics.box.map50
        map5095 = metrics.box.map
        print(f"{conf:>6.2f} | {p:>9.3f} | {r:>7.3f} | {map50:>6.3f} | {map5095:>8.3f}")


if __name__ == "__main__":
    main()

  conf | precision |  recall |  mAP50 | mAP50-95
--------------------------------------------------
Ultralytics 8.4.160  Python-3.14.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 180.848.6 MB/s, size: 15.6 KB)
val: Scanning C:\Users\EPSRG\Documents\Iftee\hostpot_detection\hotspot_dataset\test\labels.cache... 44 images, 13 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 44/44 7.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.6it/s 1.8s11.6s
                   all         44         72       0.81      0.819      0.876       0.42
Speed: 0.9ms preprocess, 7.9ms inference, 0.0ms loss, 0.4ms postprocess per image
  0.05 |     0.810 |   0.819 |  0.876 |    0.420
Ultralytics 8.4.160  Python-3.14.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO26s summary (

In [13]:
"""
Run hotspot detection on thermal images, compute the true area percentage
(correcting for overlapping boxes), and print the total coverage (or 'No hotspot') 
directly onto the manually saved image.
"""

import argparse
import cv2  
import numpy as np
from pathlib import Path
from ultralytics import YOLO


def compute_area_percent(box_xywhn):
    """
    box_xywhn: normalized (x_center, y_center, w, h), each in [0,1]
    Since w and h are already normalized to image dimensions, the box's
    area as a fraction of total image area is simply w * h.
    Used for reporting each individual box's area — still valid per-box.
    """
    _, _, w, h = box_xywhn
    return float(w * h * 100.0)


def compute_union_area_percent(boxes_xyxy, img_h, img_w):
    """
    boxes_xyxy: list of (x1, y1, x2, y2) in pixel coordinates.
    Rasterizes every box onto a shared boolean mask sized to the image and
    counts the actual covered pixels once each — this is the correct way
    to get total hotspot coverage when boxes overlap.
    """
    mask = np.zeros((img_h, img_w), dtype=bool)
    for x1, y1, x2, y2 in boxes_xyxy:
        x1, y1 = max(0, int(round(x1))), max(0, int(round(y1)))
        x2, y2 = min(img_w, int(round(x2))), min(img_h, int(round(y2)))
        mask[y1:y2, x1:x2] = True
    return float(mask.sum() / (img_h * img_w) * 100.0)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--weights", default=r"C:\Users\EPSRG\Documents\Iftee\hostpot_detection\runs\detect\yolo26s_run1\weights\best.pt")
    parser.add_argument("--source", default=r"C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set", help="image file or folder of images")
    parser.add_argument("--conf", type=float, default=0.15)
    parser.add_argument("--save-dir", default=r"C:\Users\EPSRG\Documents\Iftee\hostpot_detection\results")
    
    args = parser.parse_args(args=[])

    model = YOLO(args.weights)

    # We set save=False because we will save it manually after drawing text
    results = model.predict(
        source=args.source,
        conf=args.conf,
        save=False  
    )

    # Ensure the save directory exists
    save_dir_path = Path(args.save_dir)
    save_dir_path.mkdir(parents=True, exist_ok=True)

    for result in results:
        img_path = result.path
        print(f"\n{img_path}")

        # Get the image as a numpy array with YOLO bounding boxes already drawn on it
        annotated_img = result.plot()

        # Decide what text to draw based on detections
        if result.boxes is None or len(result.boxes) == 0:
            print("  no hotspots detected")
            text_to_draw = "No hotspot"
            color = (0, 255, 0)  # Green in BGR
        else:
            total_area_pct = 0.0
            boxes_xyxy = []
            
            for i, box in enumerate(result.boxes):
                xywhn = box.xywhn[0].tolist()   # normalized [x_c, y_c, w, h]
                xyxy = box.xyxy[0].tolist()     # pixel [x1, y1, x2, y2]
                conf = float(box.conf[0])
                
                area_pct = compute_area_percent(xywhn)
                total_area_pct += area_pct
                boxes_xyxy.append(xyxy)
                
                print(f"  hotspot {i + 1}: conf={conf:.2f}  area={area_pct:.2f}% of image")

            img_h, img_w = result.orig_shape  # (height, width) in pixels
            union_pct = compute_union_area_percent(boxes_xyxy, img_h, img_w)

            print(f"  Naive summed area (double-counts overlap): {total_area_pct:.2f}%")
            print(f"  TOTAL hotspot coverage (union, overlap-corrected): {union_pct:.2f}% of image")
            
            # Use the overlap-corrected percentage for the image text
            text_to_draw = f"Total Area: {union_pct:.2f}%"
            color = (0, 0, 255)  # Red in BGR

        # --- DRAW TEXT ON IMAGE ---
        # Get image height (h) and width (w) to calculate lower-left position
        h, w = annotated_img.shape[:2]
        
        # Position: x=20 pixels from left, y=20 pixels from bottom
        position = (20, h - 20) 
        
        # Font settings
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.0  
        thickness = 2
        
        # Draw the text onto the image array
        cv2.putText(annotated_img, text_to_draw, position, font, font_scale, color, thickness, cv2.LINE_AA)

        # Save the final image manually to your save directory
        save_file_path = save_dir_path / Path(img_path).name
        cv2.imwrite(str(save_file_path), annotated_img)
        print(f"  Saved to: {save_file_path}")


if __name__ == "__main__":
    main()


image 1/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\1min (2) (1).jpg: 320x512 2 hotspots, 28.9ms
image 2/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\1min (2) (2).jpg: 256x512 (no detections), 26.7ms
image 3/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\1min (2) (3).jpg: 256x512 2 hotspots, 26.4ms
image 4/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\1min (2).jpg: 384x512 2 hotspots, 29.9ms
image 5/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\1min (3) (1).jpg: 448x512 2 hotspots, 23.9ms
image 6/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\1min (3).jpg: 512x384 3 hotspots, 21.9ms
image 7/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\1min.jpg: 512x448 (no detections), 17.1ms
image 8/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\2min (1).jpg: 512x416 6 hotspots, 24.1ms
image 9/15 C:\Users\EPSRG\Documents\Iftee\hostpot_detection\final_set\2min (2) (1).jpg: 3

In [15]:
from ultralytics import YOLO

# 1. Load your trained model
weights_path = r"C:\Users\EPSRG\Documents\Iftee\hostpot_detection\runs\detect\yolo26s_run1\weights\best.pt"
model = YOLO(weights_path)

# 2. Run validation on your test/val dataset
# (Make sure your dataset.yaml path is correct)
data_yaml_path = r"C:\Path\To\Your\dataset.yaml"

metrics = model.val(data=r"C:\Users\EPSRG\Documents\Iftee\hostpot_detection\hotspot_dataset\data.yaml")

# 3. Print out the standard metrics
print(f"Mean Average Precision (mAP@50): {metrics.box.map50:.3f}")
print(f"Mean Average Precision (mAP@50-95): {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.p.mean():.3f}")
print(f"Recall: {metrics.box.r.mean():.3f}")

# YOLO automatically saves the Confusion Matrix image in your 'runs/detect/val' folder!

Ultralytics 8.4.160  Python-3.14.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 124.124.6 MB/s, size: 15.5 KB)
val: Scanning C:\Users\EPSRG\Documents\Iftee\hostpot_detection\hotspot_dataset\valid\labels.cache... 66 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 66/66 18.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.3s0.2ss
                   all         66        156      0.876      0.812      0.882      0.403
Speed: 1.9ms preprocess, 8.0ms inference, 0.1ms loss, 0.6ms postprocess per image
Results saved to C:\Users\EPSRG\Documents\Iftee\hostpot_detection\runs\detect\val-11
Mean Average Precision (mAP@50): 0.882
Mean Average Precision (mAP@50-95): 0.403
Precision: 0.876
Recall: 0.812
